---
## Paso 5: Optimización de Hiperparámetros

Estudio sistemático con **Grid Search manual** sobre los 5 hiperparámetros requeridos:
- Learning Rate
- Tamaño de lote
- Número de filtros convolucionales
- Tamaño de capas densas
- Número de épocas
- Tipo de dato

**Estrategia:** partiendo de los resultados del Paso 4, exploramos
combinaciones prometedoras alrededor de la mejor configuración identificada.


### 5.1 Grid Search — combinaciones de hiperparámetros

In [ ]:
# Grid de hiperparámetros a explorar
# Basado en los resultados del Paso 4:
# - batch=32 fue el mejor
# - lr=1e-3 fue el mejor
# - red pequeña funcionó mejor → probar también red mediana con más épocas

grid_configs = [
    # Configuración 1 — base del Paso 3 (referencia)
    {"nombre": "A: base",
     "batch_size": 32, "lr": 1e-3, "filtros": 32,
     "neuronas": 256, "epocas": 20, "dtype": "float32"},

    # Configuración 2 — mejor lote + red pequeña + más épocas
    {"nombre": "B: bs32-f32-n128-e25",
     "batch_size": 32, "lr": 1e-3, "filtros": 32,
     "neuronas": 128, "epocas": 25, "dtype": "float32"},

    # Configuración 3 — lr ligeramente mayor, red mediana
    {"nombre": "C: bs32-f64-n128-lr3e-4",
     "batch_size": 32, "lr": 3e-4, "filtros": 64,
     "neuronas": 128, "epocas": 25, "dtype": "float32"},

    # Configuración 4 — batch más pequeño, red pequeña
    {"nombre": "D: bs16-f32-n128",
     "batch_size": 16, "lr": 1e-3, "filtros": 32,
     "neuronas": 128, "epocas": 20, "dtype": "float32"},

    # Configuración 5 — mejor candidato según Paso 4 con más épocas
    {"nombre": "E: bs32-f32-n256-e30",
     "batch_size": 32, "lr": 1e-3, "filtros": 32,
     "neuronas": 256, "epocas": 30, "dtype": "float32"},
]

print(f"Grid Search — {len(grid_configs)} configuraciones a evaluar")
print()
print(f"{'#':<3} {'Nombre':<22} {'BS':>4} {'LR':>7} {'Filtros':>8} "
      f"{'Neuronas':>9} {'Épocas':>7}")
print("-" * 60)
for i, cfg in enumerate(grid_configs, 1):
    print(f"{i:<3} {cfg['nombre']:<22} {cfg['batch_size']:>4} "
          f"{cfg['lr']:>7.0e} {cfg['filtros']:>8} "
          f"{cfg['neuronas']:>9} {cfg['epocas']:>7}")


In [ ]:
resultados_grid = {}

for cfg in grid_configs:
    epocas = cfg["epocas"]
    hist, modelo_grid = ejecutar_experimento(cfg, epocas=epocas)
    resultados_grid[cfg["nombre"]] = hist
    # Liberar memoria
    del modelo_grid


### 5.2 Tabla comparativa — Grid Search

In [ ]:
print("=" * 82)
print("  Grid Search — Tabla Comparativa Completa")
print("=" * 82)
print(f"{'Config':<24} {'Acc.Val':>8} {'Acc.Test':>9} {'Pérd.Test':>10} "
      f"{'T.Total':>9} {'T/época':>9} {'Throughput':>12}")
print("-" * 82)

mejor_grid = None
mejor_acc  = 0.0

for nombre, h in resultados_grid.items():
    acc_t = h["exactitud_prueba"]
    print(f"{nombre:<24} "
          f"{h['exactitud_final_val']*100:>7.2f}% "
          f"{acc_t*100:>8.2f}% "
          f"{h['perdida_prueba']:>10.4f} "
          f"{h['tiempo_total']:>8.1f}s "
          f"{h['t_media_epoca']:>8.1f}s "
          f"{h['throughput']:>10.0f} img/s")
    if acc_t > mejor_acc:
        mejor_acc  = acc_t
        mejor_grid = nombre

print("=" * 82)
print(f"\n  ★ Mejor configuración : {mejor_grid}")
print(f"  ★ Exactitud prueba    : {mejor_acc*100:.2f}%")
print(f"  ★ Exactitud val       : {resultados_grid[mejor_grid]['exactitud_final_val']*100:.2f}%")


In [ ]:
# Gráfica de exactitud de validación — todas las configuraciones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
paleta = ["#2563EB", "#DC2626", "#16A34A", "#D97706", "#7C3AED"]

for (nombre, h), color in zip(resultados_grid.items(), paleta):
    ep = range(1, len(h["perdida_val"]) + 1)
    axes[0].plot(ep, h["perdida_val"], "-", color=color, lw=2, label=nombre)
    axes[1].plot(ep, [v*100 for v in h["exactitud_val"]], "-",
                 color=color, lw=2, label=nombre)

for ax, titulo, ylabel in zip(axes,
    ["Pérdida Validación", "Exactitud Validación (%)"],
    ["Pérdida", "Exactitud (%)"]):
    ax.set_xlabel("Época", fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(titulo, fontsize=11, fontweight="bold")
    ax.legend(fontsize=7.5)
    ax.grid(True, alpha=0.3)
    ax.set_facecolor("#F8FAFC")

fig.suptitle("Paso 5 — Grid Search de Hiperparámetros",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Gráfica de balance exactitud vs throughput
fig, ax = plt.subplots(figsize=(9, 5))

for (nombre, h), color in zip(resultados_grid.items(), paleta):
    ax.scatter(h["throughput"], h["exactitud_prueba"] * 100,
               s=180, color=color, zorder=5,
               label=f"{nombre}  ({h['exactitud_prueba']*100:.1f}%)")
    ax.annotate(nombre.split(":")[0].strip(),
                (h["throughput"], h["exactitud_prueba"] * 100),
                textcoords="offset points", xytext=(8, 4), fontsize=8)

ax.set_xlabel("Throughput (imágenes/s)", fontsize=11)
ax.set_ylabel("Exactitud Prueba (%)", fontsize=11)
ax.set_title("Balance Exactitud vs. Throughput — Grid Search",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=8.5)
ax.grid(True, alpha=0.3)
ax.set_facecolor("#F8FAFC")
plt.tight_layout()
plt.show()


---
## Paso 6: Análisis de Precisión Numérica

Comparar **float32 vs float16 vs bfloat16** usando la mejor configuración
identificada en el Grid Search.

Para cada dtype se analiza:
- Exactitud final
- Estabilidad numérica (NaN/Inf durante entrenamiento)
- Tiempo de entrenamiento
- Throughput
- Consumo de memoria estimado


In [ ]:
import jax.numpy as jnp

def entrenar_con_dtype(dtype_str, config_base, epocas=15):
    """
    Entrena el modelo convirtiendo los datos al dtype indicado.
    Registra NaN/Inf para evaluar estabilidad numérica.
    """
    dtype_map = {
        "float32":  jnp.float32,
        "float16":  jnp.float16,
        "bfloat16": jnp.bfloat16,
    }
    dtype = dtype_map[dtype_str]
    bs    = config_base["batch_size"]

    ds_tr  = construir_pipeline(rutas_train, etiq_train, bs, entrenamiento=True)
    ds_vl  = construir_pipeline(rutas_val,   etiq_val,   bs, entrenamiento=False)
    ds_tst = construir_pipeline(rutas_test,  etiq_test,  bs, entrenamiento=False)

    modelo_d = CNN(
        num_clases=NUM_CLASES,
        filtros=config_base["filtros"],
        neuronas=config_base["neuronas"],
        tasa_drop=0.4,
        rngs=nnx.Rngs(SEMILLA)
    )

    sched = optax.cosine_decay_schedule(
        init_value=config_base["lr"],
        decay_steps=epocas * (len(rutas_train) // bs + 1)
    )
    opt_d = nnx.Optimizer(modelo_d, optax.adam(sched))

    historial = {
        "perdida_train": [], "exactitud_train": [],
        "perdida_val":   [], "exactitud_val":   [],
        "tiempo_epoca":  [], "nan_detectado": False
    }

    print(f"\n{'='*50}")
    print(f"  dtype: {dtype_str}")
    print(f"{'='*50}")
    print(f"{'Época':>6} {'P.Train':>9} {'Acc.Train':>10} "
          f"{'P.Val':>8} {'Acc.Val':>9} {'Tiempo':>8}")
    print("-" * 53)

    t_total_ini = time.time()

    for epoca in range(1, epocas + 1):
        t0 = time.time()
        perdidas_t, exactitudes_t = [], []

        for imgs_tf, etiq_tf in ds_tr:
            # Convertir al dtype del experimento
            imgs = jnp.array(imgs_tf.numpy(), dtype=dtype)
            etiq = jnp.array(etiq_tf.numpy(), dtype=jnp.int32)
            p, a = paso_entrenamiento(modelo_d, opt_d, imgs, etiq)
            pf = float(p)
            # Detectar inestabilidad numérica
            if jnp.isnan(p) or jnp.isinf(p):
                historial["nan_detectado"] = True
                pf = float("nan")
            perdidas_t.append(pf)
            exactitudes_t.append(float(a))

        perdidas_v, exactitudes_v = [], []
        for imgs_tf, etiq_tf in ds_vl:
            imgs = jnp.array(imgs_tf.numpy(), dtype=dtype)
            etiq = jnp.array(etiq_tf.numpy(), dtype=jnp.int32)
            p, a = paso_evaluacion(modelo_d, imgs, etiq)
            perdidas_v.append(float(p))
            exactitudes_v.append(float(a))

        t_ep = time.time() - t0
        pt = np.nanmean(perdidas_t); at = np.mean(exactitudes_t)
        pv = np.nanmean(perdidas_v); av = np.mean(exactitudes_v)

        historial["perdida_train"].append(pt)
        historial["exactitud_train"].append(at)
        historial["perdida_val"].append(pv)
        historial["exactitud_val"].append(av)
        historial["tiempo_epoca"].append(t_ep)

        print(f"{epoca:>6d} {pt:>9.4f} {at*100:>9.2f}% "
              f"{pv:>8.4f} {av*100:>8.2f}% {t_ep:>7.1f}s")

    # Exactitud en prueba
    p_test, a_test = [], []
    for imgs_tf, etiq_tf in ds_tst:
        imgs = jnp.array(imgs_tf.numpy(), dtype=dtype)
        etiq = jnp.array(etiq_tf.numpy(), dtype=jnp.int32)
        p, a = paso_evaluacion(modelo_d, imgs, etiq)
        p_test.append(float(p)); a_test.append(float(a))

    historial["tiempo_total"]          = time.time() - t_total_ini
    historial["t_media_epoca"]         = np.mean(historial["tiempo_epoca"])
    historial["exactitud_final_val"]   = historial["exactitud_val"][-1]
    historial["exactitud_final_train"] = historial["exactitud_train"][-1]
    historial["exactitud_prueba"]      = float(np.mean(a_test))
    historial["perdida_prueba"]        = float(np.nanmean(p_test))
    historial["throughput"]            = len(rutas_train) / historial["t_media_epoca"]
    historial["dtype"]                 = dtype_str

    # Memoria estimada: parámetros * bytes por dtype
    bytes_dtype = {"float32": 4, "float16": 2, "bfloat16": 2}
    grafo, estado = nnx.split(modelo_d)
    n_params = sum(x.size for x in jax.tree_util.tree_leaves(estado))
    historial["memoria_mb"] = (n_params * bytes_dtype[dtype_str]) / (1024**2)

    print(f"\n  Exactitud prueba  : {historial['exactitud_prueba']*100:.2f}%")
    print(f"  NaN detectado     : {historial['nan_detectado']}")
    print(f"  Memoria estimada  : {historial['memoria_mb']:.1f} MB")

    del modelo_d
    return historial


# Usar la mejor configuración del Grid Search
config_mejor = {
    "batch_size": 32, "lr": 1e-3,
    "filtros": 32, "neuronas": 256
}
print("Configuración base para análisis de dtype:", config_mejor)


In [ ]:
resultados_dtype = {}

for dtype_str in ["float32", "float16", "bfloat16"]:
    hist = entrenar_con_dtype(dtype_str, config_mejor, epocas=15)
    resultados_dtype[dtype_str] = hist


### 6.1 Tabla comparativa — Precisión numérica

In [ ]:
print("=" * 78)
print("  Análisis de Precisión Numérica")
print("=" * 78)
print(f"{'dtype':<10} {'Acc.Val':>8} {'Acc.Test':>9} {'Estabilidad':>12} "
      f"{'T/época':>9} {'Throughput':>12} {'Mem(MB)':>8}")
print("-" * 78)

for dtype_str, h in resultados_dtype.items():
    estab = "✗ NaN/Inf" if h["nan_detectado"] else "✓ Estable"
    print(f"{dtype_str:<10} "
          f"{h['exactitud_final_val']*100:>7.2f}% "
          f"{h['exactitud_prueba']*100:>8.2f}% "
          f"{estab:>12} "
          f"{h['t_media_epoca']:>8.1f}s "
          f"{h['throughput']:>10.0f} img/s "
          f"{h['memoria_mb']:>7.1f}")

print("=" * 78)
print()

# Análisis de ventajas/desventajas
print("Ventajas y desventajas observadas:")
print()
print("  float32:")
print("    + Mayor estabilidad numérica — valor de referencia")
print("    + Rango dinámico: ±3.4×10³⁸, épsilon: 1.2×10⁻⁷")
print("    - Mayor consumo de memoria y tiempo por operación")
print()
print("  float16:")
print("    + Menor consumo de memoria (~50% vs float32)")
print("    + Mayor throughput en GPUs con Tensor Cores")
print("    - Rango limitado: ±65504 — riesgo de overflow/underflow")
print("    - Puede causar NaN si los gradientes son grandes")
print()
print("  bfloat16:")
print("    + Mismo rango que float32 (8 bits de exponente)")
print("    + Menor memoria que float32 (~50%)")
print("    + Más estable que float16 en entrenamiento profundo")
print("    - Menor precisión de mantisa (7 bits vs 23 en float32)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colores_dt = {"float32": "#2563EB", "float16": "#DC2626", "bfloat16": "#16A34A"}

for dtype_str, h in resultados_dtype.items():
    ep = range(1, len(h["perdida_val"]) + 1)
    axes[0].plot(ep, h["perdida_val"], "-", color=colores_dt[dtype_str],
                 lw=2, label=dtype_str)
    axes[1].plot(ep, [v*100 for v in h["exactitud_val"]], "-",
                 color=colores_dt[dtype_str], lw=2, label=dtype_str)

for ax, titulo, ylabel in zip(axes,
    ["Pérdida Validación", "Exactitud Validación (%)"],
    ["Pérdida", "Exactitud (%)"]):
    ax.set_xlabel("Época", fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(titulo, fontsize=11, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_facecolor("#F8FAFC")

fig.suptitle("Paso 6 — Impacto de la Precisión Numérica (float32 / float16 / bfloat16)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
dtypes   = list(resultados_dtype.keys())
colores3 = [colores_dt[d] for d in dtypes]

# Exactitud prueba
accs = [resultados_dtype[d]["exactitud_prueba"] * 100 for d in dtypes]
axes[0].bar(dtypes, accs, color=colores3, edgecolor="white", width=0.5)
axes[0].set_title("Exactitud Prueba (%)", fontweight="bold")
axes[0].set_ylim(0, 100)
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=10)

# Throughput
tputs = [resultados_dtype[d]["throughput"] for d in dtypes]
axes[1].bar(dtypes, tputs, color=colores3, edgecolor="white", width=0.5)
axes[1].set_title("Throughput (img/s)", fontweight="bold")
for i, v in enumerate(tputs):
    axes[1].text(i, v + 2, f"{v:.0f}", ha="center", fontsize=10)

# Memoria
mems = [resultados_dtype[d]["memoria_mb"] for d in dtypes]
axes[2].bar(dtypes, mems, color=colores3, edgecolor="white", width=0.5)
axes[2].set_title("Memoria estimada (MB)", fontweight="bold")
for i, v in enumerate(mems):
    axes[2].text(i, v + 0.1, f"{v:.1f}", ha="center", fontsize=10)

for ax in axes:
    ax.set_facecolor("#F8FAFC")
    ax.grid(axis="y", alpha=0.3)

fig.suptitle("Comparativa de Precisión Numérica — Exactitud · Throughput · Memoria",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


### Resumen final — Pasos 5 y 6

In [ ]:
print("=" * 60)
print("  CONFIGURACIÓN ÓPTIMA IDENTIFICADA")
print("=" * 60)

# Mejor del grid
mejor_nombre = max(resultados_grid, key=lambda k: resultados_grid[k]["exactitud_prueba"])
mejor_h      = resultados_grid[mejor_nombre]
mejor_cfg    = mejor_h["config"]

print(f"  Nombre            : {mejor_nombre}")
print(f"  Batch size        : {mejor_cfg['batch_size']}")
print(f"  Learning rate     : {mejor_cfg['lr']}")
print(f"  Filtros conv      : {mejor_cfg['filtros']} → {mejor_cfg['filtros']*2} → {mejor_cfg['filtros']*4}")
print(f"  Neuronas dense    : {mejor_cfg['neuronas']}")
print(f"  Épocas            : {mejor_cfg['epocas']}")
print(f"  Exactitud val     : {mejor_h['exactitud_final_val']*100:.2f}%")
print(f"  Exactitud prueba  : {mejor_h['exactitud_prueba']*100:.2f}%")
print(f"  Tiempo/época      : {mejor_h['t_media_epoca']:.1f} s")
print(f"  Throughput        : {mejor_h['throughput']:.0f} img/s")
print()
print("  Mejor dtype:")
mejor_dtype = max(resultados_dtype, key=lambda k: resultados_dtype[k]["exactitud_prueba"])
hd = resultados_dtype[mejor_dtype]
print(f"  dtype             : {mejor_dtype}")
print(f"  Exactitud prueba  : {hd['exactitud_prueba']*100:.2f}%")
print(f"  Estable           : {not hd['nan_detectado']}")
print(f"  Memoria           : {hd['memoria_mb']:.1f} MB")
print("=" * 60)
print()
print("  Pasos 5 y 6 completados.")
print("  Siguiente: Paso 7 — Análisis y Discusión (9 preguntas).")
